# Chapter 1: Opening the Black Box

**What you'll learn:** How to extract hidden states from a transformer, understand the layer-time grid, and run your first geometric analysis.

**Prerequisites:** Basic Python, PyTorch installed, GPU recommended.

**Time:** ~15 minutes

## 1.1 Setup

First, let's import the library and load a model. This takes ~30 seconds on first run.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Load our high-level library
import ltg

# Load the model (downloads on first run)
model = ltg.load_model("Qwen/Qwen2.5-7B", device="cuda")
print(f"Model: {model.name}")
print(f"Layers: {model.n_layers}, Hidden dim: {model.hidden_dim}")
print(f"Device: {model.device}")

## 1.2 What Are Hidden States?

A transformer processes tokens through a stack of layers. At each layer, every token has a **hidden state** — a high-dimensional vector. These vectors form a 2D grid:
- **Vertical axis (layers):** depth in the network
- **Horizontal axis (tokens):** position in the sequence

Let's extract them and see what they look like.

In [ ]:
import layer_time_geometry as core

text = "The capital of France is Paris"
H = core.extract_hidden_states(model.hf_model, model.tokenizer, text, model.device)

print(f"Hidden state tensor shape: {H.shape}")
print(f"  = ({H.shape[0]} layers, {H.shape[1]} tokens, {H.shape[2]} dimensions)")

# Get token strings for labelling
tokens = [model.tokenizer.decode([tid]) for tid in model.tokenizer.encode(text)]
print(f"\nTokens: {tokens}")

## 1.3 Visualising the Hidden-State Grid

Let's compute the norm (magnitude) of each hidden state vector and plot it as a heatmap. This gives us a first look at how the representation changes across layers and tokens.

In [ ]:
# Compute norms: ||H(l,t)|| for each cell
H_np = H[1:].cpu().numpy()  # skip layer 0 (embedding)
norms = np.linalg.norm(H_np, axis=2)  # shape (L, T)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(norms, aspect='auto', cmap='viridis', origin='lower')
ax.set_xlabel('Token position', fontsize=12)
ax.set_ylabel('Layer', fontsize=12)
ax.set_title('Hidden State Norms ||H(l,t)|| on the Layer-Time Grid', fontsize=14)

# Add token labels
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)

plt.colorbar(im, ax=ax, label='||H(l,t)||')
plt.tight_layout()
plt.show()

print(f"Norm range: {norms.min():.1f} to {norms.max():.1f}")
print(f"Notice how norms change across layers — the model transforms representations at each step.")

## 1.4 Your First Complete Analysis

The `ltg.analyse()` function does everything in one call: whitening, curvature, polar decomposition, and dependency. Let's use it and explore the results.

In [ ]:
# Complete analysis in one line
result = ltg.analyse("The capital of France is Paris", model=model)

# Print the summary
result.summary()

In [ ]:
# Generate all plots
result.plot_curvature(save_path="ch1_curvature.png")
result.plot_layer_kernel(save_path="ch1_kernel.png")
result.plot_polar(save_path="ch1_polar.png")
result.plot_dependency(save_path="ch1_dependency.png")

# Display them
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, fname, title in zip(axes.flat, 
    ['ch1_curvature.png', 'ch1_kernel.png', 'ch1_polar.png', 'ch1_dependency.png'],
    ['Curvature', 'Layer Kernel', 'Polar Decomposition', 'Dependency']):
    img = plt.imread(fname)
    ax.imshow(img)
    ax.set_title(title, fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 1.5 Comparing Two Prompts

Let's see how the geometry differs between a factual question and an arithmetic problem.

In [ ]:
r_fact = ltg.analyse("Who wrote Romeo and Juliet?", model=model)
r_math = ltg.analyse("What is 17 times 23?", model=model)

print("=== Factual Retrieval ===")
r_fact.summary()
print("\n=== Arithmetic ===")
r_math.summary()

# Side-by-side comparison
ltg.compare([r_fact, r_math], save_path="ch1_compare.png")
img = plt.imread("ch1_compare.png")
fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(img); ax.axis('off')
plt.show()

## 1.6 Key Takeaways

1. **Hidden states form a 2D grid** (layers × tokens), each cell holding a high-dimensional vector.
2. **`ltg.analyse()`** computes the full geometric decomposition in one call.
3. **Curvature peaks in the final layers** — this is where the model commits to its output.
4. **Different prompts produce different geometry** — but the overall structure (three phases) is consistent.

## Exercise

Try analysing these prompts and compare their summaries:
- `"Define photosynthesis"`
- `"If all cats are mammals and Whiskers is a cat, is Whiskers a mammal?"`
- `"Translate 'hello' to French"`

Which has the highest curvature? The lowest dependency entropy?

In [ ]:
# Your turn! Uncomment and run:
# r1 = ltg.analyse("Define photosynthesis", model=model)
# r2 = ltg.analyse("If all cats are mammals and Whiskers is a cat, is Whiskers a mammal?", model=model)
# r3 = ltg.analyse("Translate 'hello' to French", model=model)
# ltg.compare([r1, r2, r3], save_path="ch1_exercise.png")